# Fase 3 — Práctica con código
Copia cada bloque de código en tu Jupyter. Lee la explicación ANTES de ejecutar cada celda. Después de ejecutar, responde la pregunta de reflexión.

**Regla importante:** entiende cada línea antes de pasar a la siguiente.  
Si algo no funciona o no lo entiendes, escríbelo en el apartado 'Dudas' al final.

### Bloque 1 — Setup (In [1])
Configuramos los parámetros básicos. Usamos números pequeños para poder ver lo que ocurre. En un Transformer real `d_model` sería 512 y `n_heads` sería 8.

In [37]:
import numpy as np
import math

# Parámetros del Transformer (versión mini para ver los números)
d_model  = 8     # dimensión del vector de cada token (real: 512)
n_heads  = 2     # número de cabezas de atención     (real: 8)
d_k      = d_model // n_heads   # dimensión por cabeza = 4
vocab    = ['El', 'gato', 'come', 'pescado']

print(f'd_model={d_model} | n_heads={n_heads} | d_k={d_k}')

d_model=8 | n_heads=2 | d_k=4


**Pregunta 1:** ¿cuánto vale d_k si d_model=512 y n_heads=8?  
**Respuesta:** 64

**Pregunta 2:** ¿por qué usamos d_model=8 y no 512 en este notebook?  
**Respuesta:** Para ver lo que ocurre, ver un vector de 512 es más complicado

### Bloque 2 — Tokenización y Embedding (In [2])
Las palabras no son números, pero el ordenador solo entiende números. El primer paso es convertir cada palabra en un vector — una lista de números que captura su significado. Esta tabla se llama Embedding y se aprende durante el entrenamiento.  
En este notebook los vectores son aleatorios porque no estamos entrenando — solo queremos ver cómo fluyen los datos.

In [38]:
# Tabla de Embedding: una fila por token, una columna por dimensión
# En un modelo real esta tabla se APRENDE con backpropagation
np.random.seed(42)
E = np.random.randn(4, d_model)   # [4 tokens x 8 dimensiones]

frase = ['El', 'gato', 'come', 'pescado']
ids   = [0, 1, 2, 3]

# Lookup: extraer el vector de cada token usando su ID
X = E[ids]   # resultado: matriz [4 x 8]

print('Vector de cada token:')
for i, token in enumerate(frase):
    print(f'  {token:8s} (id={ids[i]}) -> {np.round(X[i], 2)}')
print(f'\nShape de X: {X.shape}  <- n_tokens x d_model')

Vector de cada token:
  El       (id=0) -> [ 0.5  -0.14  0.65  1.52 -0.23 -0.23  1.58  0.77]
  gato     (id=1) -> [-0.47  0.54 -0.46 -0.47  0.24 -1.91 -1.72 -0.56]
  come     (id=2) -> [-1.01  0.31 -0.91 -1.41  1.47 -0.23  0.07 -1.42]
  pescado  (id=3) -> [-0.54  0.11 -1.15  0.38 -0.6  -0.29 -0.6   1.85]

Shape de X: (4, 8)  <- n_tokens x d_model


**Pregunta 3:** ¿cuántos números tiene el vector de 'gato'?  
**Respuesta:** 8

**Pregunta 4:** ¿por qué 'gato' y 'perro' tendrían vectores similares en un modelo entrenado?  
**Respuesta:** porque ambos son animales y en su entrenamiento tienen contextos similares

### Bloque 3 — Positional Encoding (In [3])
El Transformer procesa todas las palabras a la vez — no las lee en orden como una RNN. Eso es muy rápido, pero tiene un problema: no sabe si 'gato' va antes o después de 'come'.  
La solución: antes de procesar nada, sumamos a cada vector una señal matemática que indica su posición en la frase. Esto se llama Positional Encoding.

In [39]:
def positional_encoding(seq_len, d_model):
    PE = np.zeros((seq_len, d_model))  # Creamos una tabla vacía de 4 filas x 8 columnas
    for pos in range(seq_len):  # Para cada posición (0, 1, 2, 3)
        for i in range(0, d_model, 2):  # Saltamos de 2 en 2 para hacer pares e impares
            denom = 10000 ** (i / d_model)
            PE[pos, i] = math.sin(pos / denom)  # Dimensión par: Seno
            PE[pos, i + 1] = math.cos(pos / denom)  # Dimensión impar: Coseno
    return PE


PE = positional_encoding(4, d_model)
X_pe = X + PE  # sumamos: mismo shape, ahora lleva posicion --> X = Significado ("gato") PE = Ubicación (Segunda Palabra)

print("Positional Encoding PE [4 x 8]:")
print(np.round(PE, 3))
print(f"\nX_pe.shape = {X_pe.shape}  <- sin cambios!")

Positional Encoding PE [4 x 8]:
[[ 0.     1.     0.     1.     0.     1.     0.     1.   ]
 [ 0.841  0.54   0.1    0.995  0.01   1.     0.001  1.   ]
 [ 0.909 -0.416  0.199  0.98   0.02   1.     0.002  1.   ]
 [ 0.141 -0.99   0.296  0.955  0.03   1.     0.003  1.   ]]

X_pe.shape = (4, 8)  <- sin cambios!


**Pregunta 5:** mira las dos primeras filas de PE. ¿Son iguales o distintas? ¿Por qué importa eso?  
**Respuesta:**
Son distintas, es importante por estos motivos:
1. Evita ambigüedad, al ser distinta distinguimos qué palabra va primero
2. Proporcionan una "huella dactilar" para cada posición
3. No solo sabemos que son distintos sino que podemos medir la distancia entre ellas.

**Pregunta 6:** ¿qué ventaja tiene usar seno y coseno en vez de simplemente usar 0, 1, 2, 3?  
**Respuesta:**
Que al usar seno y coseno siempre los valores estan entre -1 y 1, si usaramos simples numeros númericos desestabilizarían el modelo

### Bloque 4 — Proyecciones Q · K · V (In [4])
Este es el corazón del Transformer. Para calcular la atención, cada token genera tres vectores distintos:  
- **Q (Query)** — la pregunta que hace este token: ¿con qué otros tokens me relaciono?  
- **K (Key)** — la etiqueta de este token: esto es lo que yo soy  
- **V (Value)** — el contenido real de este token: esto es lo que ofrezco  

Estos tres vectores se calculan multiplicando el embedding por tres matrices (W_Q, W_K, W_V) que el modelo aprende durante el entrenamiento.

In [40]:
# Matrices de proyeccion — se APRENDEN con backpropagation
np.random.seed(7)
W_Q = np.random.randn(d_model, d_k) * 0.1   # [8 x 4]
W_K = np.random.randn(d_model, d_k) * 0.1
W_V = np.random.randn(d_model, d_k) * 0.1

# Proyectar X_pe a los tres subespacios
Q = X_pe @ W_Q   # [4 x d_k]  <- mis preguntas
K = X_pe @ W_K   # [4 x d_k]  <- mis etiquetas
V = X_pe @ W_V   # [4 x d_k]  <- mi contenido

print(f'Q.shape = {Q.shape}  (n_tokens x d_k)')
print(f'K.shape = {K.shape}')
print(f'V.shape = {V.shape}')
print('\nQ de cada token (sus preguntas):')
for i, tok in enumerate(frase):
    print(f'  {tok:8s}: {np.round(Q[i], 3)}')

Q.shape = (4, 4)  (n_tokens x d_k)
K.shape = (4, 4)
V.shape = (4, 4)

Q de cada token (sus preguntas):
  El      : [ 0.501 -0.344 -0.332 -0.489]
  gato    : [-0.116  0.168  0.091 -0.028]
  come    : [ 0.062  0.011  0.094 -0.043]
  pescado : [ 0.369 -0.096 -0.211  0.633]


**Pregunta 7:** ¿por qué Q, K y V tienen shape (4, 4) si d_model=8?  
Pista: recuerda d_k = d_model // n_heads  
**Respuesta:**
Es porque estamos preparando el terreno para la Atención Multi-Cabeza

Como definimos n_heads = 2 didividimos el trabajo para cada cabeza

$d_k = 8 // 2 = 4$.

El shape (4, 4) representa:4: El número de tokens (frase de 4 palabras).4: La dimensión reducida ($d_k$) para esa cabeza de atención específica."

### Bloque 5 — Attention weights + softmax (In [5])
Ahora calculamos cuánto presta atención cada token a los demás. Lo hacemos comparando la pregunta (Q) de cada token con la etiqueta (K) de todos los demás. Si se parecen mucho, el peso es alto.  
El resultado es una tabla donde cada fila dice: 'el token X reparte su atención así entre todos los tokens'.

In [41]:
def softmax(x):
    # Restar el maximo para estabilidad numerica
    e = np.exp(x - x.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

# 1. Puntuaciones de similitud Q * K^T
#    scores[i][j] = cuanto atiende el token i al token j
scores  = Q @ K.T / math.sqrt(d_k)   # [4 x 4]

# 2. Softmax -> pesos entre 0 y 1 que suman 1 por fila
weights = softmax(scores)             # [4 x 4]

# 3. Combinacion ponderada de los valores
out = weights @ V                     # [4 x d_k]

print('Pesos de atencion (cada fila suma 1.0):')
print(f'{" ":10s}', '  '.join(f'{t:8s}' for t in frase))
for i, tok in enumerate(frase):
    row = '  '.join(f'{w:.4f}' for w in weights[i])
    print(f'  {tok:8s}: {row}')

Pesos de atencion (cada fila suma 1.0):
           El        gato      come      pescado 
  El      : 0.1927  0.2685  0.2742  0.2646
  gato    : 0.2534  0.2554  0.2580  0.2332
  come    : 0.2426  0.2551  0.2577  0.2445
  pescado : 0.2621  0.2186  0.2176  0.3016


**Pregunta 8:** mira la fila de 'come'. ¿A qué token presta más atención?  
**Respuesta:**
A sí misma y a gato

**Pregunta 9:** ¿qué significa que la diagonal (come->come, gato->gato...) sea la más alta?  
**Respuesta:**
Significa que cada palabra es su propia funte principal de información, para entender gato lo más importante es el vector gato

**Pregunta 10:** ¿para qué sirve dividir por math.sqrt(d_k)?  
Pista: busca 'scaled dot-product attention'  
**Respuesta:**
Si los números son muy grandes la función Softmax se vuelve "rígida" (dando 1 a uno y 0 al resto) y el modelo deja de aprender

### Bloque 6 — Multi-Head Attention (In [6])
En vez de hacer una sola atención, el Transformer hace varias en paralelo — una por cabeza. Cada cabeza tiene sus propias matrices W_Q, W_K, W_V y aprende a fijarse en un tipo de relación distinta.  
Al final se concatenan los resultados de todas las cabezas y se proyectan de vuelta al tamaño original.

In [42]:
def attention_head(X, seed):
    np.random.seed(seed)
    Wq = np.random.randn(d_model, d_k) * 0.1
    Wk = np.random.randn(d_model, d_k) * 0.1
    Wv = np.random.randn(d_model, d_k) * 0.1
    Q_ = X @ Wq;  K_ = X @ Wk;  V_ = X @ Wv
    sc = softmax(Q_ @ K_.T / math.sqrt(d_k))
    return sc @ V_, sc

# Ejecutar 2 cabezas en paralelo (cada una ve algo distinto)
head1_out, head1_w = attention_head(X_pe, seed=7)
head2_out, head2_w = attention_head(X_pe, seed=99)

# Concatenar y proyectar de vuelta a d_model
multi = np.concatenate([head1_out, head2_out], axis=-1)
np.random.seed(3)
W_O       = np.random.randn(d_model, d_model) * 0.1
multi_out = multi @ W_O   # [4 x d_model]

print(f'Cabeza 1: {head1_out.shape}')
print(f'Cabeza 2: {head2_out.shape}')
print(f'Concat  : {multi.shape}')
print(f'Tras W_O: {multi_out.shape}  <- vuelve a d_model')
print('\nPesos cabeza 1 vs cabeza 2 para el token El:')
print('  H1:', np.round(head1_w[0], 2))
print('  H2:', np.round(head2_w[0], 2))

Cabeza 1: (4, 4)
Cabeza 2: (4, 4)
Concat  : (4, 8)
Tras W_O: (4, 8)  <- vuelve a d_model

Pesos cabeza 1 vs cabeza 2 para el token El:
  H1: [0.19 0.27 0.27 0.26]
  H2: [0.26 0.24 0.24 0.26]


**Pregunta 11:** los pesos de H1 y H2 para el mismo token son distintos. ¿Por qué?  
**Respuesta:**
Porque cada "cabeza" actua en paralelo y tienen opiniones diferentes, la una de la otra, cada cabeza usas sus propias matrices de proyección ($W_Q, W_K, W_V$)

**Pregunta 12:** ¿qué ventaja tiene usar 8 cabezas en vez de 1 sola?  
**Respuesta:**
1. Diversidad de enfoque, cada cabeza puede aprender ciertas cosas como aprender a buscar el sujeto, el objeto o relaciones gramaticales
2. Robustez, si una cabeza se "despista" las otras 7 pueden aportar la información correcta. Cuando juntemos la info en la matriz $W_O$ aprende a filtrar y combinar lo mejor de cada una.

### Bloque 7 — Feed-Forward + Add & LayerNorm (In [7])
Después de la atención (que mezcla información entre tokens), cada token pasa por una red neuronal pequeña de forma independiente. Esto se llama Feed-Forward Network.  
Además, en cada sub-capa se usa una conexión residual (suma la entrada al output) y una normalización (LayerNorm) para estabilizar el entrenamiento.

In [43]:
def relu(x): return np.maximum(0, x)

def layer_norm(x, eps=1e-6):
    mean = x.mean(axis=-1, keepdims=True)
    std  = x.std(axis=-1, keepdims=True)
    return (x - mean) / (std + eps)

d_ff = 16   # real: 2048 = 4 x d_model
np.random.seed(11)
W1 = np.random.randn(d_model, d_ff) * 0.1   # [8 x 16]
W2 = np.random.randn(d_ff, d_model) * 0.1   # [16 x 8]

# Sub-capa 1: atencion + residual + norm
x1 = layer_norm(X_pe + multi_out)    # Add & Norm

# Sub-capa 2: FFN + residual + norm
ffn_out = relu(x1 @ W1) @ W2
x2 = layer_norm(x1 + ffn_out)        # Add & Norm

print(f'Entrada X_pe : {X_pe.shape}')
print(f'Tras atencion: {x1.shape}  <- atencion + residual + norm')
print(f'Tras FFN     : {x2.shape}  <- FFN + residual + norm')
print('\nEl shape no cambia en ningun paso!')
print('Un bloque = Add&Norm(x + Attention(x)) + Add&Norm(x + FFN(x))')

Entrada X_pe : (4, 8)
Tras atencion: (4, 8)  <- atencion + residual + norm
Tras FFN     : (4, 8)  <- FFN + residual + norm

El shape no cambia en ningun paso!
Un bloque = Add&Norm(x + Attention(x)) + Add&Norm(x + FFN(x))


**Pregunta 13:** ¿para qué sirve sumar la entrada al output (x + f(x))? ¿Qué problema evita?  
Pista: busca 'residual connection vanishing gradient'  
**Respuesta:**
Sirve para crear una "autopista de información (Residual Connection), permitiendo que el modelo pueda tener muchisimas capas de profundidad sin que la información se pierda o se detenga el entrenamiento

**Pregunta 14:** ¿cuál es la diferencia entre lo que hace la Attention y lo que hace la FFN?  
**Respuesta:**
La diferencia es que:

Attention (Comunicación) mezcla información entre tokens, permitiendo que cada palabra entienda su contexto

FFN (Reflexión) procesa cada token de forma independiente, cada token procesa la información interna

### Bloque 8 — Encoder + Decoder completo (In [8])
El Encoder procesa la frase en español y produce representaciones ricas de cada token. El Decoder usa esas representaciones mediante Cross-Attention para generar la traducción.  
La diferencia clave: en la Cross-Attention, Q viene del Decoder (qué quiero generar), pero K y V vienen del Encoder (qué entendí del español).

In [44]:
def encoder_block(x):
    attn, _ = attention_head(x, seed=7)
    attn = np.concatenate([attn, attn], axis=-1)
    x = layer_norm(x + attn @ (np.random.randn(d_model, d_model)*0.1))
    ffn = relu(x @ W1) @ W2
    return layer_norm(x + ffn)

def decoder_block(y, encoder_kv):
    # Sub-capa 1: Masked Self-Attention (solo ve tokens anteriores)
    self_attn, _ = attention_head(y, seed=13)
    self_attn = np.pad(self_attn, [(0,0),(0,d_model-d_k)])
    y = layer_norm(y + self_attn)
    # Sub-capa 2: Cross-Attention
    #   Q del Decoder, K y V del Encoder
    np.random.seed(21)
    Wq2=np.random.randn(d_model,d_k)*0.1
    Wk2=np.random.randn(d_model,d_k)*0.1
    Wv2=np.random.randn(d_model,d_k)*0.1
    Q2=y@Wq2; K2=encoder_kv@Wk2; V2=encoder_kv@Wv2
    cross_w = softmax(Q2@K2.T/math.sqrt(d_k))
    cross_a = np.pad(cross_w@V2, [(0,0),(0,d_model-d_k)])
    y = layer_norm(y + cross_a)
    ffn = relu(y@W1)@W2
    return layer_norm(y + ffn), cross_w

# Encoder procesa el espanol
np.random.seed(0)
encoder_out = encoder_block(X_pe)   # [4 x 8]

# Decoder empieza con [START]
start = np.random.randn(1, d_model) * 0.3
decoder_out, cross_w = decoder_block(start, encoder_out)

print(f'encoder_out.shape = {encoder_out.shape}')
print(f'decoder_out.shape = {decoder_out.shape}')
print('\nCross-Attention: [START] presta atencion a:')
for i, tok in enumerate(frase):
    print(f'  {tok:8s}: {cross_w[0][i]:.3f}')

encoder_out.shape = (4, 8)
decoder_out.shape = (1, 8)

Cross-Attention: [START] presta atencion a:
  El      : 0.248
  gato    : 0.239
  come    : 0.263
  pescado : 0.250


**Pregunta 15:** ¿de dónde vienen Q, K y V en la Cross-Attention?  
**Respuesta:**

$Q$ (Query): El Decoder pregunta: "Estoy intentando generar la primera palabra de la traducción, ¿qué tengo disponible?"

$K$ (Key): El Encoder responde: "Yo tengo estas palabras en español: 'El', 'gato', 'come', 'pescado'".

$V$ (Value): El Encoder entrega el contenido: "Aquí tienes el significado de esas palabras para que las uses".

**Pregunta 16:** ¿qué significa que los pesos de cross-attention sean casi iguales para todos los tokens?  
Pista: estos pesos son sin entrenar — con entrenamiento serían muy distintos  
**Respuesta:**
El modelo no sabe en quñé palaabras debe fijarse para traducir, al ser un modelo sin entrenar se reparte la atención de forma "democrática"

### Bloque 9 — Generación autoregresiva (In [9])
El Decoder genera una palabra cada vez. Cada nueva palabra se añade a la entrada y se vuelve a pasar al Decoder. El bucle para cuando aparece el token [EOS] (End Of Sequence).

In [45]:
vocabulario_en = ['The', 'cat', 'eats', 'fish', '[EOS]']
np.random.seed(5)
W_out = np.random.randn(d_model, 5) * 0.2

generated = ['[START]']
print('Traduccion autoregressiva de "El gato come pescado":')
print()

for step in range(10):
    n = len(generated)
    np.random.seed(step * 10)
    y = np.random.randn(n, d_model) * 0.3

    dec_out, cross_w = decoder_block(y, encoder_out)

    # Proyeccion final -> probabilidades sobre vocabulario EN
    logits   = dec_out[-1] @ W_out
    probs    = softmax(logits)
    next_tok = vocabulario_en[np.argmax(probs)]

    generated.append(next_tok)
    bar = '#' * int(probs[np.argmax(probs)] * 20)
    print(f'  Paso {step+1}: {str(generated[:-1]):35s} -> "{next_tok}"  {bar}')

    if next_tok == '[EOS]': break

print()
print(f'Traduccion final: "{" ".join(generated[1:-1])}"')


Traduccion autoregressiva de "El gato come pescado":

  Paso 1: ['[START]']                         -> "eats"  #######
  Paso 2: ['[START]', 'eats']                 -> "cat"  #####
  Paso 3: ['[START]', 'eats', 'cat']          -> "The"  ####
  Paso 4: ['[START]', 'eats', 'cat', 'The']   -> "fish"  #######
  Paso 5: ['[START]', 'eats', 'cat', 'The', 'fish'] -> "The"  ######
  Paso 6: ['[START]', 'eats', 'cat', 'The', 'fish', 'The'] -> "fish"  ######
  Paso 7: ['[START]', 'eats', 'cat', 'The', 'fish', 'The', 'fish'] -> "[EOS]"  ######

Traduccion final: "eats cat The fish The fish"


**Pregunta 17:** ¿qué significa 'autoregresivo'? ¿Por qué se llama así?  
**Respuesta:**
Usa sus propias salidas anteriores como entradas para el siguiente paso.

Se llama así porque para predecir el siguiente elemento de una serie (regresión), el modelo utiliza como entrada sus propios resultados anteriores (auto). Es un proceso secuencial donde el pasado inmediato condiciona el futuro.

**Pregunta 18:** usamos np.argmax(probs) para elegir el siguiente token.  
¿Qué diferencia habría si usáramos np.random.choice(len(probs), p=probs)?  
**Respuesta:**
Al usar np.random.choice, añadirmos creatividad al modelo, con np.argmax el modelo es determinista, siempre eligiendo la palabra más problable.

Usando np.random.choice, hacemos que no se responda siempre lo mismo.

---
### Dudas
Me quedan dudas del Bloque 7 y Bloque 8 que no termino de entender, para afianzar conocimintos.



 ## 3. Entra en https://huggingface.co/models y busca un modelo de traducción ES->EN Pruébalo con pipeline('translation', model='...').

In [46]:
!pip install transformers torch sentencepiece

In [48]:
import os
from google.colab import userdata
from huggingface_hub import login

# 1. Extraer el token del panel de "Secrets" (asegúrate de que se llame exactamente HF_TOKEN)
try:
    hf_token = userdata.get('HUGGING_FACE_TOKEN')

    # 2. Establecerlo como variable de entorno para que las librerías lo vean
    os.environ["HUGGING_FACE_TOKEN"] = hf_token

    # 3. Hacer el login silencioso
    login(token=hf_token, add_to_git_credential=True)
    print("¡Token de Hugging Face configurado correctamente!")

except Exception as e:
    print("Error: Asegúrate de haber activado el acceso al secreto 'HF_TOKEN' en el panel de la izquierda.")

¡Token de Hugging Face configurado correctamente!


In [59]:
from transformers import MarianMTModel, MarianTokenizer

model_name = "Helsinki-NLP/opus-mt-es-en"

tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

frases = [
    "El gato come pescado",
    "El banco está cerrado hoy",
    "Un banco de peces nada en el mar"
]

# Tokenizar
inputs = tokenizer(frases, return_tensors="pt", padding=True)

# Generar traducción
translated = model.generate(**inputs)

# Decodificar
resultados = tokenizer.batch_decode(translated, skip_special_tokens=True)

resultados

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/826k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


...

model.safetensors:   0%|          | 0.00/312M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

['Cat eats fish', 'The bank is closed today.', 'A fish bank swims in the sea']